# Week 37, Day 4: Behaviour as data: psychometric curves, lapses and priors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Maximum likelihood with lapses

Write nll(p, c, y), the negative log-likelihood of binary choices y (1 = right) at signed contrasts c for parameters p = (mu, log sigma, gamma_L, gamma_R), returning a large number if a lapse rate leaves [0, 0.5). The fit with lapses and the fit with lapses fixed at zero are run for you.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import erf
r = np.random.default_rng(2001)
cons = np.array([-1, -0.25, -0.125, -0.0625, 0, 0.0625, 0.125, 0.25, 1])

def p_right(c, mu, sigma, gl, gr):
    return gl + (1 - gl - gr) * 0.5 * (1 + erf((c - mu) / (np.sqrt(2) * sigma)))

c = np.repeat(cons, 100)
y = (r.random(len(c)) < p_right(c, 0.02, 0.12, 0.08, 0.05)).astype(float)

def nll(p, c, y):
    mu, log_sigma, gl, gr = p
    ...

def fit(c, y, start=(0, np.log(0.2), 0.05, 0.05)):
    return minimize(nll, start, args=(c, y), method="Nelder-Mead", options={"maxiter": 3000, "xatol": 1e-5, "fatol": 1e-6}).x

p_fit = fit(c, y); sig_fit = np.exp(p_fit[1])
p0 = minimize(lambda q: nll([q[0], q[1], 0, 0], c, y), [0, np.log(0.2)], method="Nelder-Mead").x
sig_nolapse = np.exp(p0[1])
print(f"with lapses: bias {p_fit[0]:+.3f}, threshold {sig_fit:.3f}, lapses {p_fit[2]:.3f} / {p_fit[3]:.3f}")
print(f"without lapses: threshold {sig_nolapse:.3f}")
cc = np.linspace(-1, 1, 400)
plt.plot(cons, [y[c == k].mean() for k in cons], "ok", label="data")
plt.plot(cc, p_right(cc, p_fit[0], sig_fit, p_fit[2], p_fit[3]), label="with lapses")
plt.plot(cc, p_right(cc, p0[0], sig_nolapse, 0, 0), "--", label="without lapses")
plt.xlabel("signed contrast"); plt.ylabel("P(right)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fit with lapses recovers the threshold to within 0.05 and both lapse rates to within 0.05"
    assert abs(sig_fit - 0.12) < 0.05 and abs(p_fit[2] - 0.08) < 0.05 and abs(p_fit[3] - 0.05) < 0.05

def test_no_lapse():
    "ignoring lapses at least doubles the fitted threshold"
    assert sig_nolapse > 2 * sig_fit

test_fit()
test_no_lapse()
print("All checks passed")

### Exercise: A bootstrap interval for the threshold

Repeat 100 times: resample 100 trials with replacement within each contrast (keeping contrasts fixed), refit with fit(), and store the threshold. Set ci to the 2.5th and 97.5th percentiles. (This takes a few seconds.)

In [ ]:
boot = []
for b in range(100):
    idx = ...
    boot.append(np.exp(fit(c[idx], y[idx], start=p_fit)[1]))
ci = np.percentile(boot, [2.5, 97.5])
print(f"threshold {sig_fit:.3f}, 95% bootstrap interval {ci[0]:.3f} to {ci[1]:.3f}")
plt.hist(boot, 20); plt.axvline(0.12, color="k", ls="--"); plt.xlabel("bootstrap threshold"); plt.show()

In [ ]:
import numpy as np
def test_ci():
    "the 95% interval contains the true threshold 0.12 and is narrower than 0.12"
    assert ci[0] < 0.12 < ci[1] and ci[1] - ci[0] < 0.12

test_ci()
print("All checks passed")

### Exercise: The prior shifts the curve

Fit each block's data with fit(), then compute P(right) at zero contrast from each fit, storing them in at_zero['right'] and at_zero['left'].

In [ ]:
cb = np.repeat(cons, 80)
y_rb = (r.random(len(cb)) < p_right(cb, -0.08, 0.12, 0.06, 0.06)).astype(float)    # right-biased blocks
y_lb = (r.random(len(cb)) < p_right(cb, 0.08, 0.12, 0.06, 0.06)).astype(float)     # left-biased blocks
fit_r = ...
fit_l = ...
at_zero = {"right": p_right(0, fit_r[0], np.exp(fit_r[1]), fit_r[2], fit_r[3]),
           "left": p_right(0, fit_l[0], np.exp(fit_l[1]), fit_l[2], fit_l[3])}
print(f"bias: right blocks {fit_r[0]:+.3f}, left blocks {fit_l[0]:+.3f}")
print(f"P(right) at zero contrast: right blocks {at_zero['right']:.2f}, left blocks {at_zero['left']:.2f}")
for name, f_ in (("right blocks", fit_r), ("left blocks", fit_l)):
    plt.plot(cc, p_right(cc, f_[0], np.exp(f_[1]), f_[2], f_[3]), label=name)
plt.xlabel("signed contrast"); plt.ylabel("P(right)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shift():
    "the bias in right blocks is below the bias in left blocks by at least 0.08"
    assert fit_l[0] - fit_r[0] > 0.08

def test_zero():
    "at zero contrast, P(right) is at least 0.25 higher in right blocks"
    assert at_zero["right"] - at_zero["left"] > 0.25

test_shift()
test_zero()
print("All checks passed")